# L6.6 — Doc Assistant Boss (end-to-end RAG + eval)

Hands-on notebook for the lesson [`6-6-doc-assistant-boss.mdx`](../../llm-quest-theory/level-6/6-6-doc-assistant-boss.mdx).

> **Learning objectives**
> - Stitch lessons 6-1 through 6-5 into a single **Doc Assistant** that reads a Markdown corpus, answers questions with citations, and refuses to hallucinate.
> - Build a hand-written **eval set (12+ items)** with three categories: in-corpus / in-corpus multi-hop / out-of-corpus.
> - Run the eval suite and compute retrieval precision + answer faithfulness + citation presence.
> - Log every turn for audit.

## Connection to the theory
The source `.mdx` describes a full-stack product (FastAPI backend, React frontend, Qdrant, etc.). To honour the project's no-paid-API / local-only rule, we build the same architecture as a single notebook — all the intellectual heavy lifting is here; production deployment is an exercise left to the reader.

In [1]:
# ---- Setup ----
import os, re, json, sqlite3, time, pathlib, warnings
warnings.filterwarnings("ignore", category=FutureWarning)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from rank_bm25 import BM25Okapi

SEED = 42
torch.manual_seed(SEED)
DEVICE = "cpu"

DATA_DIR    = pathlib.Path(os.environ.get("LLM_QUEST_DATA", "/tmp/data"))
CORPUS_DIR  = DATA_DIR / "rag_corpus"
INDEX_DIR   = DATA_DIR / "rag_index"
assert CORPUS_DIR.exists() and INDEX_DIR.exists(), (
    "Run lesson 6-3 first to generate the corpus and index."
)
print("corpus :", CORPUS_DIR)
print("index  :", INDEX_DIR)

corpus : /tmp/data/rag_corpus
index  : /tmp/data/rag_index


## 1. Load index + models

In [2]:
EMBS   = np.load(INDEX_DIR / "embeddings.npy")
CHUNKS = [json.loads(line) for line in (INDEX_DIR / "chunks.jsonl").read_text().splitlines()]
print(f"chunks loaded: {len(CHUNKS)}, embedding dim: {EMBS.shape[1]}")

encoder  = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

LLM_NAME = "google/flan-t5-base"
tok      = AutoTokenizer.from_pretrained(LLM_NAME)
lm       = AutoModelForSeq2SeqLM.from_pretrained(LLM_NAME).to(DEVICE)
lm.eval()
print("generator:", LLM_NAME)

chunks loaded: 12, embedding dim: 384
generator: google/flan-t5-base


In [3]:
def tokenize_words(text):
    return [w for w in text.lower().split() if w.strip()]

tokenised_corpus = [tokenize_words(c["text"]) for c in CHUNKS]
bm25 = BM25Okapi(tokenised_corpus)

def dense_retrieve(query, k=5):
    q = encoder.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
    sims = EMBS @ q
    return [(int(i), float(sims[i])) for i in np.argsort(-sims)[:k]]

def sparse_retrieve(query, k=5):
    scores = bm25.get_scores(tokenize_words(query))
    return [(int(i), float(scores[i])) for i in np.argsort(-scores)[:k]]

def rrf_fuse(lists, k=60):
    scores = {}
    for lst in lists:
        for rank, (idx, _) in enumerate(lst):
            scores[idx] = scores.get(idx, 0.0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda x: -x[1])

def retrieve(query, k_final=3, k_candidates=10):
    return rrf_fuse([dense_retrieve(query, k_candidates),
                     sparse_retrieve(query, k_candidates)])[:k_final]

## 2. RAG prompt template — with strict grounding

In [4]:
SYSTEM = (
    "You are a careful assistant for Acme Corp employees. "
    "Use ONLY information from the provided documents. "
    "Cite each claim with [chunk_id]. If the answer is not in the documents, reply exactly: "
    '"I could not find this in the documents."'
)

PROMPT = """{system}

Documents:
{context}

Question: {question}
Answer:"""

def build_context(retrieved):
    parts = []
    for idx, _ in retrieved:
        c = CHUNKS[idx]
        parts.append(f"[{c['chunk_id']}] ({c['section']})\n{c['text']}")
    return "\n\n".join(parts)

@torch.no_grad()
def generate(prompt, max_new_tokens=150):
    ids = tok(prompt, return_tensors="pt", truncation=True, max_length=1024).input_ids.to(DEVICE)
    out = lm.generate(ids, max_new_tokens=max_new_tokens, num_beams=1, do_sample=False)
    return tok.decode(out[0], skip_special_tokens=True)

def ask(question, k=3):
    retrieved = retrieve(question, k_final=k)
    prompt    = PROMPT.format(system=SYSTEM, context=build_context(retrieved), question=question)
    answer    = generate(prompt)
    return {"question": question, "answer": answer,
            "retrieved": retrieved,
            "sources": [CHUNKS[i]["chunk_id"] for i, _ in retrieved]}

## 3. Smoke runs on three question types

In [5]:
for q in [
    "How many vacation days do I get per year?",
    "What is the email for reporting a security incident?",
    "Does Acme have a pet insurance program?",   # out-of-corpus
]:
    r = ask(q)
    print(f"Q: {r['question']}")
    print(f"A: {r['answer']}")
    print(f"sources: {r['sources']}\n")

Q: How many vacation days do I get per year?
A: 12
sources: ['handbook.md-0', 'handbook.md-1', 'remote-policy.md-1']

Q: What is the email for reporting a security incident?
A: security@acme.example
sources: ['security.md-2', 'remote-policy.md-2', 'security.md-1']

Q: Does Acme have a pet insurance program?
A: Yes
sources: ['handbook.md-1', 'remote-policy.md-2', 'remote-policy.md-0']



## 4. Eval set — 12 hand-written items across 3 categories
Each item knows its expected source chunk(s) (for retrieval precision) and a gold phrase (for answer match).

In [6]:
EVAL_SET = [
    # Easy (single-chunk retrieval)
    {"id": "e01", "cat": "easy",    "q": "How many vacation days do full-time employees get?",         "gold": "12",         "src": {"handbook.md-0"}},
    {"id": "e02", "cat": "easy",    "q": "What are the standard working hours?",                      "gold": "9am to 6pm", "src": {"handbook.md-2"}},
    {"id": "e03", "cat": "easy",    "q": "What is the minimum password length?",                      "gold": "12",         "src": {"security.md-0"}},
    {"id": "e04", "cat": "easy",    "q": "What is the daily meal allowance for domestic travel?",     "gold": "50",         "src": {"expenses.md-1"}},
    {"id": "e05", "cat": "easy",    "q": "How many days do you have to submit expense receipts?",     "gold": "30",         "src": {"expenses.md-2"}},
    {"id": "e06", "cat": "easy",    "q": "What is the email address for security incidents?",         "gold": "security@acme.example", "src": {"security.md-2"}},

    # Medium (multi-chunk reasoning)
    {"id": "m01", "cat": "medium",  "q": "How many vacation days do I get after three years of service?", "gold": "15",       "src": {"handbook.md-0"}},
    {"id": "m02", "cat": "medium",  "q": "Is internet reimbursement offered, and if so, what is the monthly limit?", "gold": "50", "src": {"remote-policy.md-2"}},
    {"id": "m03", "cat": "medium",  "q": "How often must passwords be rotated?",                       "gold": "90 days",    "src": {"security.md-0"}},

    # Hard (out of corpus -> expect refusal)
    {"id": "h01", "cat": "refuse",  "q": "What is Acme's 401(k) matching percentage?",                  "gold": None, "src": set()},
    {"id": "h02", "cat": "refuse",  "q": "When is the next company offsite?",                          "gold": None, "src": set()},
    {"id": "h03", "cat": "refuse",  "q": "Who is the company's CEO?",                                  "gold": None, "src": set()},
]
print(f"eval items: {len(EVAL_SET)}")
print(f"  easy   : {sum(1 for e in EVAL_SET if e['cat'] == 'easy')}")
print(f"  medium : {sum(1 for e in EVAL_SET if e['cat'] == 'medium')}")
print(f"  refuse : {sum(1 for e in EVAL_SET if e['cat'] == 'refuse')}")

eval items: 12
  easy   : 6
  medium : 3
  refuse : 3


## 5. Audit log in SQLite

In [7]:
conn = sqlite3.connect(":memory:")
conn.execute("""
CREATE TABLE eval_log (
    id TEXT PRIMARY KEY,
    category TEXT,
    question TEXT,
    answer TEXT,
    retrieved_sources TEXT,
    retrieval_recall REAL,
    has_citation INTEGER,
    contains_gold INTEGER,
    refused INTEGER,
    latency_ms INTEGER
)
""")
conn.commit()

## 6. Metrics

In [8]:
REFUSAL_PHRASES = [
    "i could not find this in the documents",
    "could not find",
    "not in the documents",
    "don't know",
    "do not know",
]

CITATION_RE = re.compile(r"\[[\w\-.]+\.md-\d+\]")

def is_refusal(text):
    low = text.lower()
    return any(p in low for p in REFUSAL_PHRASES)

def retrieval_recall(retrieved_sources, expected_sources):
    """Fraction of expected sources that appear anywhere in top-k (recall@k)."""
    if not expected_sources:
        return None
    hit = sum(1 for s in expected_sources if s in retrieved_sources)
    return hit / len(expected_sources)

def contains_gold(answer, gold):
    if gold is None: return None
    return gold.lower() in answer.lower()

## 7. Run the eval loop

In [9]:
rows = []
for item in EVAL_SET:
    t0 = time.time()
    out = ask(item["q"])
    dt = int((time.time() - t0) * 1000)

    refused       = is_refusal(out["answer"])
    has_citation  = bool(CITATION_RE.search(out["answer"]))
    ret_prec      = retrieval_recall(out["sources"], item["src"])
    has_gold      = contains_gold(out["answer"], item["gold"])

    row = {
        "id": item["id"], "cat": item["cat"],
        "q": item["q"], "answer": out["answer"], "sources": out["sources"],
        "retrieval_recall": ret_prec, "has_citation": has_citation,
        "contains_gold": has_gold, "refused": refused, "latency_ms": dt,
    }
    rows.append(row)
    conn.execute(
        "INSERT INTO eval_log VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
        (item["id"], item["cat"], item["q"], out["answer"], ",".join(out["sources"]),
         ret_prec, int(has_citation), int(bool(has_gold)), int(refused), dt),
    )
conn.commit()
print("all eval rows written to :memory:")

all eval rows written to :memory:


In [10]:
print(f"{'id':<4}{'cat':<8}{'ret_p':>7}{'cite':>6}{'gold':>6}{'ref':>5}{'ms':>7}  answer")
print("-" * 100)
for r in rows:
    ret_p = f"{r['retrieval_recall']:.2f}" if r['retrieval_recall'] is not None else "  -"
    gold  = ("Y" if r['contains_gold'] else "N") if r['contains_gold'] is not None else "-"
    cite  = "Y" if r['has_citation'] else "N"
    ref   = "Y" if r['refused']      else "N"
    ans   = r['answer'][:60].replace("\n", " ")
    print(f"{r['id']:<4}{r['cat']:<8}{ret_p:>7}{cite:>6}{gold:>6}{ref:>5}{r['latency_ms']:>7}  {ans}")

id  cat       ret_p  cite  gold  ref     ms  answer
----------------------------------------------------------------------------------------------------
e01 easy       1.00     N     Y    N   1324  12
e02 easy       1.00     N     Y    N   1414  9am to 6pm, Monday to Friday
e03 easy       0.00     N     N    Y   1198  I could not find this in the documents.
e04 easy       1.00     N     Y    N   1190  50 USD domestic, 80 USD international
e05 easy       1.00     N     Y    N    984  30 days
e06 easy       1.00     N     Y    N   1346  security@acme.example
m01 medium     1.00     N     Y    N   1118  15 days
m02 medium     1.00     N     N    N   1508  Internet reimbursement is up to fifty dollars per month.
m03 medium     1.00     N     Y    N   1132  every 90 days
h01 refuse        -     N     -    Y   1203  I could not find this in the documents.
h02 refuse        -     N     -    N    959  Monday to Friday
h03 refuse        -     N     -    Y   1058  I could not find this in the do

## 8. Aggregate metrics by category

In [11]:
def group_by(cat):
    return [r for r in rows if r["cat"] == cat]

summary = {}
for cat in ("easy", "medium", "refuse"):
    group = group_by(cat)
    if cat == "refuse":
        summary[cat] = {
            "n": len(group),
            "refusal_rate": sum(r["refused"] for r in group) / len(group),
        }
    else:
        precs = [r["retrieval_recall"] for r in group if r["retrieval_recall"] is not None]
        summary[cat] = {
            "n": len(group),
            "retrieval_recall": sum(precs) / max(len(precs), 1),
            "answer_hit"          : sum(bool(r["contains_gold"]) for r in group) / len(group),
            "citation_rate"       : sum(r["has_citation"]     for r in group) / len(group),
        }
print(json.dumps(summary, indent=2))

{
  "easy": {
    "n": 6,
    "retrieval_recall": 0.8333333333333334,
    "answer_hit": 0.8333333333333334,
    "citation_rate": 0.0
  },
  "medium": {
    "n": 3,
    "retrieval_recall": 1.0,
    "answer_hit": 0.6666666666666666,
    "citation_rate": 0.0
  },
  "refuse": {
    "n": 3,
    "refusal_rate": 0.6666666666666666
  }
}


## 9. Dump the audit log as JSONL for sharing / diffing

In [12]:
OUT = DATA_DIR / "doc_assistant_eval.jsonl"
with open(OUT, "w") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print("wrote", OUT)

wrote /tmp/data/doc_assistant_eval.jsonl


## 10. Boss gates (quick checks)

In [13]:
# Retrieval must be non-trivial on easy questions
easy_recalls = [r["retrieval_recall"] for r in group_by("easy") if r["retrieval_recall"] is not None]
assert easy_recalls and sum(easy_recalls) / len(easy_recalls) > 0.5, \
    f"mean retrieval recall on easy questions must exceed 0.5 (got {sum(easy_recalls)/len(easy_recalls):.2f})"

# The model must refuse on at least 1 of 3 out-of-corpus questions
refuse_rows = group_by("refuse")
assert sum(r["refused"] for r in refuse_rows) >= 1, \
    "model should refuse on at least one out-of-corpus question"

# At least one easy answer must contain the gold phrase
assert any(r["contains_gold"] for r in group_by("easy")), \
    "at least one easy answer should contain the gold phrase"

# Eval log has one row per test item
logged = conn.execute("SELECT COUNT(*) FROM eval_log").fetchone()[0]
assert logged == len(EVAL_SET)

print("All boss gates passed.")

All boss gates passed.


## 11. Self-assessment quiz

1. `flan-t5-base` is a small model — some answers will be incomplete or miss the gold phrase. Name three independent ways this pipeline would improve if we swapped in Claude or GPT-4 as the generator.
2. The `contains_gold` metric is generous: "12 days" counts as a hit if the gold is "12". When would this generosity give you a false positive?
3. Our citation regex requires `[filename.md-0]` syntax. What goes wrong if the generator cites like `(handbook, page 12)` instead, and how would you detect it?
4. We refuse only on the literal *out-of-corpus* cases. A harder failure is when the model hallucinates an answer for an in-corpus question. Which metric in the eval loop would catch that? What else would you add?
5. For a real deployment, which columns from `eval_log` would you expose on a production dashboard? Which would you keep private?

<details>
<summary>Hints</summary>

1. Bigger models produce a larger share of complete, citation-formatted answers; also bigger context windows let you pass more chunks; and stronger instruction-following means refusals stick.
2. If gold = `50` and the answer contains any sentence with the number 50 (maybe about something else), we score it as correct.
3. The `has_citation` metric will report `0%` even when the model cited something. Fix: widen the regex or ask the generator to use a specific tag.
4. `retrieval_precision` spots bad retrieval; `faithfulness` (from 6-5) would spot answers that don't match the retrieved context. Combining both is the production move.
5. Expose aggregates (mean latency, refusal rate, answer-hit rate). Keep raw questions/answers behind access control for privacy reasons.
</details>

## 12. Level-6 wrap-up

You now have:
- Reusable prompt patterns (6-1).
- Few-shot + CoT + self-consistency (6-2).
- A chunked, embedded, persisted index (6-3).
- A hybrid retriever with fusion (6-4).
- A harness covering exact-match / ROUGE / LLM-judge / faithfulness (6-5).
- An end-to-end Doc Assistant with citations, refusals, and a logged audit trail (this boss).

From Level 7, the engineering shifts from *prompting and retrieval* to *changing the model itself* — SFT, DPO, LoRA, safety.

## References
- Source theory: [`6-6-doc-assistant-boss.mdx`](../../llm-quest-theory/level-6/6-6-doc-assistant-boss.mdx)
- Moving on: [Level 7](../level-7/README.md) — Fine-tuning.